# ADT Contracts

```{contents}
:local:
:depth: 2
```

An **abstract data type** starts with behavior. Before deciding whether data should be stored in an array, linked nodes, or a hash table, we decide what operations the type promises to support.


```{index} Describe the Observable History First
```



```{index} Describe the Observable History First
```

## Describe the Observable History First



An undo history is not defined by an array or a linked node. It is defined by what a client can observe after a sequence of operations. If the client records A, records B, and then undoes once, it expects B to be removed before A. An implementation can store those entries in different ways while preserving that behavior. This separates the abstract data type, the behavioral contract, from the data structure used to realize it.

Write the history on paper before writing a class. For each operation, record the result, the resulting abstract contents, and the empty/failure policy. If the contract says an unsuccessful TryPop leaves state unchanged, returning false after already decrementing Count violates the contract even if later tests happen to restore the count.

| Operation | Result | Abstract history, oldest to newest |
| --- | --- | --- |
| Start | Empty | [] |
| Push A | Recorded | [A] |
| Push B | Recorded | [A, B] |
| TryPop | Success, B | [A] |
| TryPop | Success, A | [] |
| TryPop | Failure | [] |

The empty failure is part of the behavior, not an implementation detail to settle later. A different ADT might throw on an empty Pop. Both can be coherent contracts, but a client written for one cannot assume the other. Also specify whether null entries, duplicates, or a fixed capacity are allowed. These policies determine which tests should succeed and what failure must preserve.

For related explanation, see the further reading below.[^development-reading]


```{index} Separate Client Knowledge from Implementation Knowledge
```



```{index} Separate Client Knowledge from Implementation Knowledge
```

## Separate Client Knowledge from Implementation Knowledge



A C# interface can express operation names and types, for example a success-returning method with an out result. It cannot by itself express every ordering rule or state invariant. Two classes can compile against the same interface and still behave differently. Tests and explanations connect the syntax to the promised history.

The client should not need to know the backing array's capacity or the head node's field name. If a capacity change affects returned values or exposes unused slots, representation details have leaked through the abstraction. Conversely, the implementer must know enough about storage to maintain its invariants. Abstraction assigns different responsibilities to the two roles; it does not mean that implementation details are unimportant.

Guided exercise: keep the interface unchanged but change removal to return A before B. Identify the first row at which this behavior differs from the specified history. Then explain why a test that checks only Count would miss the defect. Extend the history with a failed removal followed by a new push; this checks whether failure left the structure reusable.

```{dropdown} Contract checkpoint
The first successful removal must return B. A queue-like implementation could keep the right counts while returning the wrong element. Checking returned values, order, state after failure, and later reuse tests more of the contract than checking size alone.
```


```{index} abstract data type
```
```{index} contract; data structure
```



```{index} What an ADT Specifies
```

## What an ADT Specifies



An ADT describes a set of values and the operations allowed on those values. It does **not** require one particular internal representation.

For a stack, the contract might say:

| Operation | Meaning |
| --- | --- |
| `Push(item)` | Add an item to the top. |
| `Pop()` | Remove and return the top item. |
| `Peek()` | Return the top item without removing it. |
| `Count` | Report how many items are stored. |

This contract says nothing about arrays or linked lists. That separation is the point: client code should depend on the behavior it needs, not on accidental implementation details.


```{index} interface; ADT
```



```{index} Interfaces as Contracts
```

## Interfaces as Contracts



In C#, an `interface` is a natural way to write an ADT contract. It names the operations but leaves the storage strategy to a class.


In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
IStack<string> history = new ListStack<string>();
history.Push("open file");
history.Push("rename variable");
Console.WriteLine(history.Pop());
Console.WriteLine(history.Peek());

interface IStack<T>
{
    int Count { get; }
    void Push(T item);
    T Pop();
    T Peek();
}

class ListStack<T> : IStack<T>
{
    private readonly List<T> items = new();
    public int Count => items.Count;

    public void Push(T item) => items.Add(item);

    public T Pop()
    {
        if (items.Count == 0) throw new InvalidOperationException("Stack is empty.");
        int last = items.Count - 1;
        T value = items[last];
        items.RemoveAt(last);
        return value;
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Stack is empty.");
        return items[^1];
    }
}

rename variable
open file


```{index} client code
```



```{index} Client Code Depends on Behavior
```

## Client Code Depends on Behavior



**Client code** is code that uses a type. If client code only needs stack behavior, it should not care whether the stack is backed by a `List<T>`, an array, or linked nodes.

That means this is a better variable type for many methods:

```csharp
static void UndoLastAction(IStack<string> actions)
```

than this:

```csharp
static void UndoLastAction(ListStack<string> actions)
```

The first version accepts any stack implementation that satisfies the contract.


```{index} invariant
```



```{index} Contracts Include Rules
```

## Contracts Include Rules



A contract includes more than method names. It also includes rules:

- `Pop` removes the same item that `Peek` would return.
- `Count` increases after `Push` and decreases after `Pop`.
- `Pop` and `Peek` on an empty stack are errors.

A rule that must remain true for the object to be valid is called an **invariant**. Data-structure code often exists to preserve invariants while operations run quickly.


```{index} Practice
```

## Practice



1. Write a contract for a queue ADT using operation names and meanings.
2. Name one invariant a queue implementation must preserve.
3. Explain why a method parameter typed as `IStack<string>` is more flexible than one typed as `ListStack<string>`.


```{index} Summary
```

## Summary



An ADT names behavior. An interface can express that behavior in C#. Implementations provide storage and algorithms behind the contract. This separation lets programs change representation without changing every caller.


```{rubric} Footnotes
```


[^1301_adt_contracts-1]: Some C# library types use classes directly instead of exposing a small custom interface. The ADT idea still helps you reason about what behavior the code depends on.


[^development-reading]: Further reading: [OpenDSA, Lists: Chapter Introduction](https://opendsa.org/OpenDSA/Books/Catalog/html/ListIntro.html). The worked business scenarios and guided reasoning here are original teaching examples.
